In [405]:
from spacenet.point_patterns.helpers.compute_contributions import compute_contributions
from spacenet.point_patterns.helpers.compute_contributions_parallel import compute_contributions_parallel   
from spacenet.point_patterns.helpers.spatial_bootstrap import spatial_bootstrap
from spacenet.point_patterns.helpers.is_connected_filter import is_connected_filter
from spacenet.helpers.node_node_distance import node_node_distance  
from spacenet.point_patterns.helpers.polynomial_kernel_bandwidth_scale import polynomial_kernel_bandwidth_scale

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import squidpy as sq
from scipy.ndimage import gaussian_filter
from skimage.color import label2rgb
from skimage.draw import ellipse
from skimage.measure import label, regionprops, regionprops_table
from skimage.transform import rotate
import warnings

from scipy.stats import kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from skimage.segmentation import expand_labels, watershed
import os
import spacenet as sn
import random

import warnings
warnings.filterwarnings('ignore')


from spacenet.point_patterns.helpers.polynomial_kernel_bandwidth_scale import polynomial_kernel_bandwidth_scale

In [ ]:
### This below calculates the PCF function across a certain distance "r_max" for each TMA sample
### You just need to upload your DF and delegate the target FOV or name, along with the cluster or target of interest columns

In [ ]:
### Now lets compare our ROI's to 100 random patches WITHIN THE TMA of the same size
### I am now expanding it to compare all patches in this TMA with 1000 random samples around the TMA
### Then determining if 'IN GENERAL' the patches of this TMA are more or less clustered
from joblib import Parallel, delayed
import contextlib, io

BANDWIDTH = 120  # your fixed, deliberately-chosen value

def process_fov(fov, df):
    try:
        points = np.array([df['centroid_x'], df['centroid_y']]).T
        cat_labels = df['cluster'].astype(str)
        spatial_net = sn.utils.spatial_network_from_points(points)
        sn.utils.add_node_labels(spatial_net, labels=cat_labels, node_label_name='Cell Clusters')

        with contextlib.redirect_stdout(io.StringIO()):
            radii, pcf_values, con_inter = sn.point_patterns.cross_pair_correlation_function(
                spatial_net,
                spatial_kernel_bandwidth=BANDWIDTH,
                r_max=500,
                r_step=10,
                return_confidence_interval=True
            )
        return {"TMA": fov, "PCF": pcf_values, "radii": radii, "con_inter": con_inter}
    except RuntimeError as e:
        print(f"Skipped {fov}: {e}")
        return None

all_results = Parallel(n_jobs=4, verbose=0)(
    delayed(process_fov)(fov, df[df['fov'] == fov])
    for fov in df['fov'].unique()
)

results = pd.DataFrame([r for r in all_results if r is not None])

Computing contributions:  62%|██████▏   | 8995/14423 [12:06<07:48, 11.59contributions/s]/s]